# Courier Delivery Performance Analysis
### Data Analytics Project | Information Technology | Chaitanya Bharathi Institute of Technology (A)

**Dataset:** `train.csv` — courier/food delivery records  
**Project focus:** Explore delivery time and investigate how traffic, weather, vehicle condition, city, order type, and delivery-person characteristics relate to delivery performance.

> **Important:** Keep `train.csv` in the same folder as this notebook before running it. The supplied file contains 45,593 records and 20 columns. This is a large dataset by row count (not column count).

## Course Outcome Mapping

- **CO1 — NumPy Basics** (Unit I)
- **CO2 — Getting Started with Pandas; Data Loading, Storage and File Formats** (Unit II)
- **CO3 — Data Cleaning and Preparation; Data Wrangling** (Unit III)
- **CO4 — Data Aggregation and Group Operations; Time Series** (Unit IV)
- **CO5 — Plotting and Visualization** (Unit V)

Run the notebook from top to bottom. Some results can vary slightly if the dataset or library versions differ.

## Project Objectives
1. Load and inspect the delivery dataset.
2. Clean inconsistent text, missing values, and delivery-time values.
3. Engineer useful features such as order hour, delivery distance, and delivery-time bands.
4. Compare delivery performance across traffic, weather, cities, vehicle types, and other factors.
5. Visualize findings and summarize practical recommendations.

## Business Questions
- What is the typical delivery time, and how variable is it?
- Which traffic and weather conditions are associated with longer delivery times?
- Do delivery times differ by city, vehicle type, or vehicle condition?
- How does approximate delivery distance relate to delivery time?
- Are delivery times different at various hours of the day?

# CO1 — NumPy Basics (Unit I)
This section demonstrates ndarray creation, data types, arithmetic, indexing and slicing, Boolean and fancy indexing, transposing/swapping axes, pseudo-random number generation, and universal functions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

np.random.seed(42)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("train.csv")
if not DATA_PATH.exists() and Path("/mnt/data/train.csv").exists():
    DATA_PATH = Path("/mnt/data/train.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find train.csv. Place the supplied CSV in the same folder as this notebook."
    )

raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Dataset shape:", raw.shape)
display(raw.head())

In [ ]:
# 1. Creating ndarrays and checking data types
delivery_times_demo = np.array([24, 33, 26, 21, 30], dtype=np.int64)
ratings_demo = np.array([4.9, 4.5, 4.4, 4.7, 4.6], dtype=np.float64)

print("Delivery-time array:", delivery_times_demo)
print("Shape:", delivery_times_demo.shape, "| dtype:", delivery_times_demo.dtype)
print("Ratings array:", ratings_demo, "| dtype:", ratings_demo.dtype)

# 2. Arithmetic with NumPy arrays
print("Delivery times + 5 minutes:", delivery_times_demo + 5)
print("Delivery times in seconds:", delivery_times_demo * 60)
print("Mean delivery time:", np.mean(delivery_times_demo))
print("Difference from mean:", delivery_times_demo - np.mean(delivery_times_demo))

# 3. Basic indexing and slicing
print("First delivery time:", delivery_times_demo[0])
print("First three delivery times:", delivery_times_demo[:3])
print("Every second value:", delivery_times_demo[::2])

# 4. Boolean indexing and fancy indexing
print("Times above 25 minutes:", delivery_times_demo[delivery_times_demo > 25])
print("Values at positions 0, 2, and 4:", delivery_times_demo[[0, 2, 4]])

# 5. Transposing arrays and swapping axes
matrix_demo = np.array([[24, 33, 26], [21, 30, 28]])
print("Original matrix shape:", matrix_demo.shape)
print(matrix_demo)
print("Transposed matrix shape:", matrix_demo.T.shape)
print(matrix_demo.T)
array_3d = np.arange(24).reshape(2, 3, 4)
print("3D shape before swap:", array_3d.shape)
print("3D shape after swapping axes 0 and 2:", np.swapaxes(array_3d, 0, 2).shape)

# 6. Pseudo-random number generation (demonstration only; not actual observed data)
sample_simulated_times = np.random.randint(15, 51, size=10)
print("Simulated sample for demonstration:", sample_simulated_times)

# 7. Universal functions
print("Square roots:", np.sqrt(delivery_times_demo))
print("Absolute differences from 27 minutes:", np.abs(delivery_times_demo - 27))
print("Rounded ratings:", np.round(ratings_demo))

In [ ]:
# Apply NumPy to the real dataset's delivery-time target after extracting its numeric value.
target_numeric = pd.to_numeric(
    raw["Time_taken(min)"].astype(str).str.extract(r"(-?\d+(?:\.\d+)?)", expand=False),
    errors="coerce"
).to_numpy(dtype=float)

valid_target = target_numeric[np.isfinite(target_numeric)]
print("Valid numeric delivery-time observations:", valid_target.size)
print("Mean:", np.nanmean(valid_target))
print("Median:", np.nanmedian(valid_target))
print("Standard deviation:", np.nanstd(valid_target))
print("25th, 50th, and 75th percentiles:", np.nanpercentile(valid_target, [25, 50, 75]))

# CO2 — Getting Started with Pandas and Data Loading (Unit II)
We inspect Series, DataFrame and Index objects; perform reindexing, dropping, filtering, arithmetic/alignment, function application, sorting/ranking, duplicate-label checks, descriptive statistics, chunked reading, and CSV writing.

In [ ]:
print("Rows and columns:", raw.shape)
print("\nColumn names:")
print(raw.columns.tolist())
print("\nData types:")
display(raw.dtypes.rename("dtype").to_frame())
print("\nDataFrame index:", type(raw.index).__name__)
print("Example Series type:", type(raw["City"]).__name__)
display(raw.describe(include="all").T)

In [ ]:
# Series, DataFrame and Index objects
city_series = raw["City"]
print("Series sample:")
display(city_series.head())
print("DataFrame sample:")
display(raw[["City", "Road_traffic_density", "Time_taken(min)"]].head())
print("Index sample:", raw.index[:5].tolist())

# Reindexing and dropping entries from an axis (demonstration on a small copy)
small = raw.head(5).copy()
reindexed = small.reindex([0, 1, 2, 3, 4, 10])
print("Reindexed sample (new row becomes missing):")
display(reindexed.tail(2))
print("Drop a column from a copy:")
display(small.drop(columns=["ID"]).head(2))

# Indexing, selection, filtering
display(raw.loc[raw["City"].astype(str).str.strip().eq("Urban")].head(3))
display(raw.iloc[:3, :5])

# Arithmetic and data alignment
left = pd.Series([10, 20, 30], index=["A", "B", "C"])
right = pd.Series([1, 2, 3], index=["B", "C", "D"])
print("Series arithmetic aligns by index:")
display(left + right)

# Function application and mapping
display(raw[["City", "Type_of_vehicle"]].head().apply(lambda col: col.astype(str).str.strip()))
display(raw["City"].head().map(lambda value: str(value).strip().title()))

# Sorting and ranking
time_preview = pd.DataFrame({"delivery_time": valid_target[:10]})
time_preview["rank"] = time_preview["delivery_time"].rank(method="dense")
display(time_preview.sort_values("delivery_time", ascending=False))

# Duplicate labels (axis labels) and duplicate rows are different checks
print("Duplicate column labels:", raw.columns.duplicated().sum())
print("Duplicate index labels:", raw.index.duplicated().sum())
print("Duplicate complete rows:", raw.duplicated().sum())

# Writing a small text/CSV output
raw.head(100).to_csv("courier_sample_output.csv", index=False)
print("Wrote courier_sample_output.csv (first 100 original records).")

In [ ]:
# Read text/CSV data in pieces (chunks). This counts rows without loading all data into memory.
chunk_rows = 0
chunk_city_counts = {}
for chunk in pd.read_csv(DATA_PATH, chunksize=10000, low_memory=False):
    chunk_rows += len(chunk)
    if "City" in chunk.columns:
        counts = chunk["City"].astype(str).str.strip().value_counts()
        for city, count in counts.items():
            chunk_city_counts[city] = chunk_city_counts.get(city, 0) + int(count)

print("Rows counted using chunked reading:", chunk_rows)
display(pd.Series(chunk_city_counts, name="record_count").sort_values(ascending=False).to_frame())

# CO3 — Data Cleaning, Preparation and Wrangling (Unit III)
We standardize text fields, parse delivery times, inspect missing values and duplicates, fill or retain missing values appropriately, transform/map/replace values, create bins and categorical variables, and demonstrate hierarchical indexing, joins, merges, concatenation, overlap combination, and reshaping.

In [ ]:
df = raw.copy()

# Strip whitespace from object/string columns and normalize common placeholder values.
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({
        "": pd.NA, "NaN": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA
    })

# Extract numeric target from strings such as '(min) 24'
df["Time_taken(min)"] = pd.to_numeric(
    df["Time_taken(min)"].astype("string").str.extract(r"(-?\d+(?:\.\d+)?)", expand=False),
    errors="coerce"
)

# Convert numeric-like fields
numeric_cols = [
    "Delivery_person_Age", "Delivery_person_Ratings", "Vehicle_condition",
    "multiple_deliveries", "Restaurant_latitude", "Restaurant_longitude",
    "Delivery_location_latitude", "Delivery_location_longitude"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Shape before row-level cleaning:", df.shape)
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate complete rows:", df.duplicated().sum())

In [ ]:
# Missing-data handling: preserve missingness for analysis; use a median only for selected
# numerical fields in a separate modeling/summary copy. Avoid inventing target delivery times.
df = df.drop_duplicates().copy()
for col in ["City", "Weatherconditions", "Road_traffic_density", "Type_of_order",
            "Type_of_vehicle", "Festival"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

analysis_df = df.copy()
for col in ["Delivery_person_Age", "Delivery_person_Ratings", "Vehicle_condition",
            "multiple_deliveries"]:
    analysis_df[col] = analysis_df[col].fillna(analysis_df[col].median())

# Do not impute the target outcome; exclude missing/implausible target values for target analysis.
analysis_df = analysis_df[
    analysis_df["Time_taken(min)"].notna()
    & analysis_df["Time_taken(min)"].between(1, 180)
].copy()

print("Cleaned rows:", len(analysis_df))
print("Missing values after selected handling:")
display(analysis_df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Delivery-time summary:")
display(analysis_df["Time_taken(min)"].describe().to_frame())

In [ ]:
# Transforming with functions/mapping and replacing values
analysis_df["City"] = analysis_df["City"].astype("string").str.title()
analysis_df["Road_traffic_density"] = analysis_df["Road_traffic_density"].replace({
    "Jam": "Jam", "Low": "Low", "Medium": "Medium", "High": "High"
})
analysis_df["Weatherconditions"] = analysis_df["Weatherconditions"].astype("string").str.replace(
    r"^conditions\s*", "", regex=True
).str.title()

# Renaming axis labels
renamed_preview = analysis_df.head(5).rename(columns={
    "Time_taken(min)": "Delivery_Time_Minutes",
    "Delivery_person_Ratings": "Delivery_Person_Rating"
})
display(renamed_preview)

# Discretization / binning
analysis_df["Delivery_Time_Band"] = pd.cut(
    analysis_df["Time_taken(min)"],
    bins=[0, 20, 30, 40, 60, 180],
    labels=["<=20 min", "21-30 min", "31-40 min", "41-60 min", ">60 min"],
    include_lowest=True
)
analysis_df["Distance_Band"] = pd.cut(
    np.sqrt(
        ((analysis_df["Delivery_location_latitude"] - analysis_df["Restaurant_latitude"]) * 111) ** 2
        + ((analysis_df["Delivery_location_longitude"] - analysis_df["Restaurant_longitude"]) * 111
           * np.cos(np.deg2rad(analysis_df["Restaurant_latitude"]))) ** 2
    ),
    bins=[-0.01, 2, 5, 10, 20, np.inf],
    labels=["0-2 km", "2-5 km", "5-10 km", "10-20 km", ">20 km"]
)

# Approximate straight-line distance (not road distance)
lat1 = np.deg2rad(analysis_df["Restaurant_latitude"])
lat2 = np.deg2rad(analysis_df["Delivery_location_latitude"])
dlat = lat2 - lat1
dlon = np.deg2rad(analysis_df["Delivery_location_longitude"] - analysis_df["Restaurant_longitude"])
a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
analysis_df["Approx_Distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))

# Categorical extension type and categorical computations
for col in ["City", "Road_traffic_density", "Weatherconditions", "Type_of_vehicle",
            "Type_of_order", "Festival", "Delivery_Time_Band"]:
    analysis_df[col] = analysis_df[col].astype("category")

display(analysis_df[["Time_taken(min)", "Delivery_Time_Band", "Approx_Distance_km",
                     "Distance_Band", "City", "Road_traffic_density"]].head())
print("Categorical city counts:")
display(analysis_df["City"].value_counts(dropna=False).to_frame("records"))

In [ ]:
# Detect/filter unusual values using IQR rule (flag only; do not automatically delete)
q1 = analysis_df["Time_taken(min)"].quantile(0.25)
q3 = analysis_df["Time_taken(min)"].quantile(0.75)
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
analysis_df["Delivery_Time_IQR_Outlier"] = ~analysis_df["Time_taken(min)"].between(lower_bound, upper_bound)
print(f"IQR bounds: {lower_bound:.2f} to {upper_bound:.2f} minutes")
print("Flagged outlier count:", int(analysis_df["Delivery_Time_IQR_Outlier"].sum()))
display(analysis_df.loc[analysis_df["Delivery_Time_IQR_Outlier"],
                        ["Time_taken(min)", "City", "Road_traffic_density"]].head(10))

# Hierarchical indexing, reordering/sorting levels and summary by level
multi = analysis_df.set_index(["City", "Type_of_vehicle"]).sort_index()
print("MultiIndex example:")
display(multi[["Time_taken(min)", "Delivery_person_Ratings"]].head(8))
print("Summary statistics by City level:")
display(multi.groupby(level="City", observed=True)["Time_taken(min)"].agg(["count", "mean", "median", "std"]))

# Indexing with DataFrame columns / database-style joins / merging on index
city_summary = analysis_df.groupby("City", observed=True).agg(
    city_mean_delivery=("Time_taken(min)", "mean"),
    city_order_count=("ID", "count")
)
joined = analysis_df.join(city_summary, on="City")
display(joined[["City", "Time_taken(min)", "city_mean_delivery", "city_order_count"]].head())

vehicle_summary = analysis_df.groupby("Type_of_vehicle", observed=True)["Time_taken(min)"].mean().rename(
    "vehicle_mean_delivery"
).to_frame()
merged = analysis_df.merge(vehicle_summary, left_on="Type_of_vehicle", right_index=True, how="left")
display(merged[["Type_of_vehicle", "Time_taken(min)", "vehicle_mean_delivery"]].head())

# Concatenating along an axis
part_a, part_b = analysis_df.head(3), analysis_df.iloc[3:6]
display(pd.concat([part_a, part_b], axis=0).shape)
side_by_side = pd.concat([
    analysis_df[["ID", "City"]].head(5).reset_index(drop=True),
    analysis_df[["Time_taken(min)", "Road_traffic_density"]].head(5).reset_index(drop=True)
], axis=1)
display(side_by_side)

# Combining data with overlap (combine_first)
left_demo = pd.DataFrame({"rating": [4.5, np.nan, 4.2]}, index=["A", "B", "C"])
right_demo = pd.DataFrame({"rating": [4.1, 4.7, 4.8]}, index=["A", "B", "C"])
display(left_demo.combine_first(right_demo))

# Reshaping with hierarchical indexing: pivot / unstack
pivot_city_traffic = analysis_df.pivot_table(
    index="City", columns="Road_traffic_density", values="Time_taken(min)",
    aggfunc="mean", observed=True
)
display(pivot_city_traffic)
display(pivot_city_traffic.stack().rename("mean_delivery_minutes").to_frame().head(10))

# CO4 — Data Aggregation, Group Operations and Time Series (Unit IV)
This section uses column-wise and multiple aggregations, split-apply-combine, aggregation without row indexes, date/time parsing, date ranges, time-based indexing, duplicate time-index demonstrations, resampling, downsampling, upsampling, and interpolation.

In [ ]:
# Column-wise and multiple function application
traffic_summary = analysis_df.groupby("Road_traffic_density", observed=True).agg(
    orders=("ID", "count"),
    mean_delivery_min=("Time_taken(min)", "mean"),
    median_delivery_min=("Time_taken(min)", "median"),
    std_delivery_min=("Time_taken(min)", "std"),
    mean_rating=("Delivery_person_Ratings", "mean")
).sort_values("mean_delivery_min", ascending=False)
display(traffic_summary)

# General split-apply-combine and aggregation without row index
city_vehicle_summary = analysis_df.groupby(
    ["City", "Type_of_vehicle"], observed=True, as_index=False
).agg(
    orders=("ID", "count"),
    average_delivery_minutes=("Time_taken(min)", "mean"),
    median_delivery_minutes=("Time_taken(min)", "median")
)
display(city_vehicle_summary.head(15))

# Time series: convert date and time strings, with safe parsing
analysis_df["Order_Date_Parsed"] = pd.to_datetime(
    analysis_df["Order_Date"], format="%d-%m-%Y", errors="coerce"
)
analysis_df["Order_Hour"] = pd.to_numeric(
    analysis_df["Time_Orderd"].astype("string").str.extract(r"^(\d{1,2}):", expand=False),
    errors="coerce"
)
analysis_df["Order_Datetime"] = pd.to_datetime(
    analysis_df["Order_Date_Parsed"].dt.strftime("%Y-%m-%d") + " " +
    analysis_df["Time_Orderd"].astype("string"),
    errors="coerce"
)
print("Parsed order dates:", analysis_df["Order_Date_Parsed"].notna().sum())
print("Parsed order datetimes:", analysis_df["Order_Datetime"].notna().sum())
display(analysis_df[["Order_Date", "Time_Orderd", "Order_Date_Parsed", "Order_Datetime", "Order_Hour"]].head())

In [ ]:
# Time-series indexing, selection and subsetting
dated = analysis_df.dropna(subset=["Order_Date_Parsed"]).copy()
daily = dated.groupby("Order_Date_Parsed")["Time_taken(min)"].agg(["count", "mean", "median"]).sort_index()
daily.index.name = "Order_Date"
display(daily.head())

if not daily.empty:
    print("First date in dataset:", daily.index.min())
    print("Last date in dataset:", daily.index.max())
    # Date range and frequency demonstration
    calendar_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    print("Number of calendar dates in range:", len(calendar_range))
    daily_full = daily.reindex(calendar_range)
    daily_full.index.name = "Order_Date"
    print("Dates with no observed orders:", int(daily_full["count"].isna().sum()))
    display(daily_full.head())

    # Resampling/downsampling: weekly mean of daily average delivery time
    weekly = daily["mean"].resample("W").mean()
    print("Weekly average of daily mean delivery time:")
    display(weekly.head(10))

    # Upsampling and interpolation: illustrative daily time series only.
    # Interpolated values are estimates, not observed delivery records.
    daily_mean = daily["mean"].asfreq("D")
    upsampled_interpolated = daily_mean.interpolate(method="time")
    print("Upsampled daily mean with time interpolation (first 10):")
    display(upsampled_interpolated.head(10))
else:
    print("No valid dates available for time-series analysis.")

# Duplicate time indices demonstration: multiple orders can occur on the same date.
duplicate_date_count = dated["Order_Date_Parsed"].duplicated(keep=False).sum()
print("Rows whose date is shared by multiple orders:", int(duplicate_date_count))

In [ ]:
# Time-of-day analysis
hour_summary = analysis_df.dropna(subset=["Order_Hour"]).groupby("Order_Hour").agg(
    orders=("ID", "count"),
    mean_delivery_min=("Time_taken(min)", "mean"),
    median_delivery_min=("Time_taken(min)", "median")
).sort_index()
display(hour_summary)

# Compare festival / multiple-delivery conditions
if "Festival" in analysis_df.columns:
    display(analysis_df.groupby("Festival", observed=True)["Time_taken(min)"].agg(
        ["count", "mean", "median"]
    ).sort_values("mean", ascending=False))
if "multiple_deliveries" in analysis_df.columns:
    display(analysis_df.groupby("multiple_deliveries")["Time_taken(min)"].agg(
        ["count", "mean", "median"]
    ).sort_index())

# CO5 — Plotting and Visualization (Unit V)
Charts use Matplotlib and Seaborn. Each chart includes readable labels and titles. These plots are exploratory: association does not prove that a factor causes delivery delays.

In [ ]:
# 1. Matplotlib figure/subplots: delivery-time distribution and boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(analysis_df["Time_taken(min)"].dropna(), bins=30, edgecolor="black")
axes[0].set_title("Distribution of Delivery Time")
axes[0].set_xlabel("Delivery time (minutes)")
axes[0].set_ylabel("Number of orders")

axes[1].boxplot(analysis_df["Time_taken(min)"].dropna(), vert=True)
axes[1].set_title("Delivery Time — Boxplot")
axes[1].set_ylabel("Delivery time (minutes)")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Seaborn bar plot: average delivery time by traffic density
traffic_order = ["Low", "Medium", "High", "Jam"]
traffic_plot = analysis_df.copy()
traffic_plot["Road_traffic_density"] = traffic_plot["Road_traffic_density"].astype(str)
traffic_plot = traffic_plot[traffic_plot["Road_traffic_density"].isin(traffic_order)]
plt.figure(figsize=(9, 5))
sns.barplot(data=traffic_plot, x="Road_traffic_density", y="Time_taken(min)",
            order=traffic_order, hue="Road_traffic_density", errorbar=None, palette="deep", legend=False)
plt.title("Average Delivery Time by Road Traffic Density")
plt.xlabel("Road traffic density")
plt.ylabel("Average delivery time (minutes)")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Line plot: average delivery time by order hour
if not hour_summary.empty:
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=hour_summary.reset_index(), x="Order_Hour", y="mean_delivery_min",
                 marker="o", label="Mean delivery time")
    plt.xticks(range(0, 24, 1))
    plt.title("Average Delivery Time by Order Hour")
    plt.xlabel("Order hour")
    plt.ylabel("Average delivery time (minutes)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 4. Histogram and density plot
plt.figure(figsize=(10, 5))
sns.histplot(data=analysis_df, x="Time_taken(min)", bins=30, kde=True, color="steelblue")
plt.title("Delivery Time Histogram with Density Curve")
plt.xlabel("Delivery time (minutes)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# 5. Scatter/point plot: approximate straight-line distance vs delivery time
distance_plot = analysis_df.dropna(subset=["Approx_Distance_km", "Time_taken(min)"]).copy()
distance_plot = distance_plot[distance_plot["Approx_Distance_km"].between(0, 60)]
# Sample points for plotting speed; analysis remains based on the full cleaned dataset.
distance_sample = distance_plot.sample(n=min(5000, len(distance_plot)), random_state=42)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=distance_sample, x="Approx_Distance_km", y="Time_taken(min)",
                alpha=0.45, s=25, color="steelblue")
plt.title("Approximate Straight-Line Distance vs Delivery Time")
plt.xlabel("Approximate distance (km)")
plt.ylabel("Delivery time (minutes)")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Categorical comparison: vehicle type
vehicle_summary_plot = analysis_df.groupby("Type_of_vehicle", observed=True).agg(
    mean_delivery=("Time_taken(min)", "mean"),
    order_count=("ID", "count")
).reset_index().sort_values("mean_delivery", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=vehicle_summary_plot, x="Type_of_vehicle", y="mean_delivery",
            errorbar=None, hue="Type_of_vehicle", palette="deep", legend=False, ax=axes[0])
axes[0].set_title("Average Delivery Time by Vehicle Type")
axes[0].set_xlabel("Vehicle type")
axes[0].set_ylabel("Mean delivery time (minutes)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=vehicle_summary_plot, x="Type_of_vehicle", y="order_count", hue="Type_of_vehicle", palette="deep", legend=False, ax=axes[1])
axes[1].set_title("Number of Orders by Vehicle Type")
axes[1].set_xlabel("Vehicle type")
axes[1].set_ylabel("Number of orders")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Facet Grid: delivery time by traffic, split by city (top cities by order count)
top_cities = analysis_df["City"].value_counts().head(3).index.tolist()
facet_df = analysis_df[analysis_df["City"].isin(top_cities)].copy()
facet_df["City"] = facet_df["City"].astype(str)
facet_df["Road_traffic_density"] = facet_df["Road_traffic_density"].astype(str)

if not facet_df.empty:
    g = sns.catplot(
        data=facet_df, x="Road_traffic_density", y="Time_taken(min)",
        col="City", kind="box", col_wrap=3, height=4, aspect=1.2,
        order=["Low", "Medium", "High", "Jam", "Unknown"]
    )
    g.set_axis_labels("Traffic density", "Delivery time (minutes)")
    g.set_titles("{col_name}")
    g.fig.suptitle("Delivery Time by Traffic Density Across Top Cities", y=1.03)
    plt.show()

In [ ]:
# Save a chart to a file (example)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=traffic_plot, x="Road_traffic_density", y="Time_taken(min)",
            order=traffic_order, hue="Road_traffic_density", errorbar=None, palette="deep", legend=False, ax=ax)
ax.set_title("Average Delivery Time by Traffic Density")
ax.set_xlabel("Road traffic density")
ax.set_ylabel("Average delivery time (minutes)")
fig.tight_layout()
fig.savefig("courier_traffic_delivery_time.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved courier_traffic_delivery_time.png")

# Final Results and Findings
The cells below generate a summary from the cleaned dataset rather than hard-coding conclusions. Use the printed results and charts to write evidence-based findings in your submission.

In [ ]:
print("FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Original records: {len(raw):,}")
print(f"Cleaned records used for analysis: {len(analysis_df):,}")
print(f"Records excluded during duplicate/target-quality cleaning: {len(raw) - len(analysis_df):,}")
print(f"Mean delivery time: {analysis_df['Time_taken(min)'].mean():.2f} minutes")
print(f"Median delivery time: {analysis_df['Time_taken(min)'].median():.2f} minutes")
print(f"Standard deviation: {analysis_df['Time_taken(min)'].std():.2f} minutes")

print("\nAverage delivery time by traffic:")
display(analysis_df.groupby("Road_traffic_density", observed=True)["Time_taken(min)"]
        .agg(["count", "mean", "median"]).sort_values("mean", ascending=False))

print("\nAverage delivery time by weather:")
display(analysis_df.groupby("Weatherconditions", observed=True)["Time_taken(min)"]
        .agg(["count", "mean", "median"]).sort_values("mean", ascending=False))

print("\nAverage delivery time by city:")
display(analysis_df.groupby("City", observed=True)["Time_taken(min)"]
        .agg(["count", "mean", "median"]).sort_values("mean", ascending=False))

print("\nCorrelation among selected numeric fields (correlation is not causation):")
corr_cols = ["Time_taken(min)", "Delivery_person_Age", "Delivery_person_Ratings",
             "Vehicle_condition", "multiple_deliveries", "Approx_Distance_km"]
display(analysis_df[corr_cols].corr(numeric_only=True).round(2))

## Conclusion (complete after running the notebook)
Use the generated tables and plots to complete these points:

1. **Overall performance:** Report the mean and median delivery time and explain what they imply.
2. **Traffic and weather:** Identify the groups with the highest average delivery time, while checking their sample sizes.
3. **Operational factors:** Compare delivery time by vehicle type, vehicle condition, city, and number of simultaneous deliveries.
4. **Distance and time:** Describe the visible relationship between approximate straight-line distance and delivery time. State that straight-line distance is only a proxy for road distance.
5. **Recommendations:** Suggest operational improvements supported by the results (for example, traffic-aware planning or reviewing multiple-delivery assignments).
6. **Limitations:** This dataset is observational; associations do not establish causality. Some fields may be missing or inconsistently recorded. The dataset covers a limited date range and the calculated coordinate distance does not follow actual roads.

## Libraries Used
- **NumPy:** numerical arrays, calculations, random sampling, universal functions.
- **Pandas:** tabular analysis, cleaning, grouping, joins, reshaping, and time series.
- **Matplotlib:** figures, subplots, labels, legends, and saving plots.
- **Seaborn:** statistical plots and categorical comparisons.

## How to Run
1. Put this notebook and `train.csv` in the same folder.
2. Install dependencies if needed: `pip install numpy pandas matplotlib seaborn jupyter`.
3. Open Jupyter Notebook or JupyterLab.
4. Run all cells from top to bottom.
5. Review the generated tables and charts before writing the final project conclusions.